In [ ]:
# Native libraries
from random import randrange
from typing import Callable, Iterable
import os

# Thirdy-Party Libraries
from bs4 import BeautifulSoup
from bs4 import ResultSet, Tag
from httpx import Response
from httpx import AsyncClient
import httpx
import pandas as pd


In [ ]:
# Simple User-Agent Rotation

user_agents: list[str] = [
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Ubuntu Chromium/37.0.2062.94 Chrome/37.0.2062.94 Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/45.0.2454.85 Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.1; WOW64; rv:40.0) Gecko/20100101 Firefox/40.0',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_5) AppleWebKit/600.8.9 (KHTML, like Gecko) Version/8.0.8 Safari/600.8.9',
    'Mozilla/5.0 (iPad; CPU OS 8_4_1 like Mac OS X) AppleWebKit/600.1.4 (KHTML, like Gecko) Version/8.0 Mobile/12H321 Safari/600.1.4',
]

get_random_user_agent: Callable = lambda: user_agents[randrange(len(user_agents))]

In [ ]:
# Base Utilities

base_url: str = 'https://blogbbm.com/manga/'
base_response: Response = httpx.get(url=base_url, headers={'User-Agent': get_random_user_agent()})

base_soup: BeautifulSoup = BeautifulSoup(base_response.text, 'html.parser')

# Just checking
base_response, base_response.url

In [ ]:
# Retrieving base informations

rows: ResultSet[Tag] = base_soup.select_one('tbody').select('tr')

table_data: list[dict[str, str]] = []

for row in rows:
    title, author, publisher, demography, year = row.select('td')

    table_data.append(
        {
            'url': title.select_one('a').get('href'),
            'title': title.text,
            'author': author.text,
            'publisher': publisher.text,
            'demography': demography.text,
            'year': year.text,
        }
    )

# Just Checking
table_data[:3]

In [ ]:
# Structuring Data
base_data: pd.DataFrame = pd.DataFrame(table_data)

# Just Checking
base_data

In [ ]:
# Just Checking
base_data.describe()

In [ ]:
# Just Checking
base_data.info()

In [ ]:
# Helper function

async def download_image(url: str, save_dir: str) -> None:

    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    image_name: list[str] = url.split('/')[-1]
    save_path: str = os.path.join(save_dir, image_name)

    async with httpx.AsyncClient() as client:
        response: Response = await client.get(url)
        response.raise_for_status()

        with open(save_path, "wb") as file:
            file.write(response.content)

    print(f"Image has been saved. Path: {save_path}")

In [ ]:



urls: Iterable[str] = base_data.query('title in ("One Piece",)')['url']
header_contents: list[dict[str, str]] = []
table_contents: list[dict[str, str]] = []

for url in urls:
    response: Response = httpx.get(url, headers={'User-Agent': get_random_user_agent()})
    soup: BeautifulSoup = BeautifulSoup(response.text, 'html.parser')

    header_strongs = soup.select_one('.entry-content p').select('strong')
    header_contents.append(
        {strong.text: strong.next_sibling.strip(': \n') for strong in header_strongs}
    )


    image_paths = [tag.get('src') for tag in soup.select('div.gallery figure img')]

    for idx, path in enumerate(image_paths):
        await download_image(path, f'img/One-Piece')

    tables = soup.select('table')

    for table in tables:
        rows = table.select('tbody tr')

        for idx, row in enumerate(rows):
            try:
                vol, japan, brazil, price, buy_in = row.select('td')

                if idx == 0:
                    continue

                buy_url_tag = buy_in.select('a')
                buy_url = None if not buy_url_tag else [href.get('href') for href in buy_url_tag]

                table_contents.append(
                    {
                        'volume': vol.text,
                        'japan': japan.text.replace('\n', ' | '),
                        'brazil': brazil.text,
                        'price': price.text,
                        'buy_in': buy_in.text.replace('\n', ' & '),
                        'url': buy_url
                    }
                )
            except:
                continue


In [ ]:
image_paths

In [ ]:
base_data.head(30)


In [ ]:
pd.DataFrame(header_contents)

In [ ]:

pd.DataFrame(table_contents).head(30)
